In [ ]:
%load_ext autoreload
%autoreload 2

# Table Extraction Comparison: BAML vs Docling

This notebook compares two different approaches to extracting structured information from medical PDF tables:

1. **BAML (Semantic Approach)**: PDF → Images → LLM Analysis → Semantic Triples
2. **Docling (Structural Approach)**: PDF → OCR/Table Detection → Row/Column Data

We'll analyze the results from different PDF pages containing medical guideline tables. Use the dropdown below to select which example to analyze.

In [ ]:
import json
import pandas as pd
from pathlib import Path
import fitz  # PyMuPDF
from IPython.display import Image, display, HTML
import matplotlib.pyplot as plt
from typing import Dict, List, Any, Tuple, Union
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output

# Configuration for different examples
EXAMPLES = {
    "Page 37 - Medical Imaging Methods": {
        "pdf": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/pdf_pages/_37.pdf",
        "docling": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/docling/_37/tables/tables_summary.json",
        "baml": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/table_structures/from_pdf_images/_37.json"
    },
    "Page 8 - Clinical Guidelines": {
        "pdf": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/pdf_pages/_8.pdf",
        "docling": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/docling/_8/tables/tables_summary.json",
        "baml": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/table_structures/from_pdf_images/_8.json"
    },
    "Page 50 - Treatment Protocols": {
        "pdf": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/pdf_pages/_50.pdf",
        "docling": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/docling/_50/tables/tables_summary.json",
        "baml": "/home/pwiesenbach/CardioGuidelinesGraph/scripts_emre/data/guidelines/table_structures/from_pdf_images/_50.json"
    }
}

# Create dropdown widget
example_dropdown = widgets.Dropdown(
    options=list(EXAMPLES.keys()),
    value=list(EXAMPLES.keys())[0],
    description='Example:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Global variables for current paths
current_paths = {}

def update_paths(change):
    """Update file paths based on dropdown selection."""
    global current_paths
    selected_example = change['new']
    current_paths = EXAMPLES[selected_example].copy()
    print(f"📄 Selected: {selected_example}")
    print(f"📄 PDF: {current_paths['pdf']}")
    print(f"🔍 Docling: {current_paths['docling']}")
    print(f"🤖 BAML: {current_paths['baml']}")

# Set initial paths
update_paths({'new': example_dropdown.value})

# Connect dropdown to update function
example_dropdown.observe(update_paths, names='value')

# Display the dropdown
display(example_dropdown)
print("✅ Example selector ready - choose an example above to analyze")

In [ ]:
def load_json_file(file_path: str) -> Union[Dict[str, Any], List[Any]]:
    """Load and return JSON file contents."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return {}
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error in {file_path}: {e}")
        return {}

def display_pdf_page(pdf_path: str, page_num: int = 0):
    """Display a page from the PDF as an image."""
    try:
        if not Path(pdf_path).exists():
            print(f"❌ PDF file not found: {pdf_path}")
            return
            
        doc = fitz.open(pdf_path)
        if page_num >= len(doc):
            print(f"❌ Page {page_num} not found in PDF (total pages: {len(doc)})")
            doc.close()
            return
            
        page = doc[page_num]
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for better quality
        img_data = pix.tobytes("png")
        doc.close()
        
        display(Image(data=img_data))
        print(f"📄 Displaying page {page_num + 1} of {Path(pdf_path).name}")
    except Exception as e:
        print(f"❌ Error displaying PDF: {e}")

def load_current_data():
    """Load data for currently selected example."""
    if not current_paths:
        print("❌ No example selected")
        return {}, {}
    
    docling_data = load_json_file(current_paths['docling'])
    baml_data = load_json_file(current_paths['baml'])
    
    print("✅ Files loaded successfully")
    print(f"📊 Docling data keys: {list(docling_data.keys()) if isinstance(docling_data, dict) else 'Not a dict'}")
    print(f"🤖 BAML data: {len(baml_data) if isinstance(baml_data, list) else 'Not a list'} items")
    
    # Check data structure
    if docling_data:
        print(f"📊 Docling data type: {type(docling_data)}")
    if baml_data:
        print(f"🤖 BAML data type: {type(baml_data)}")
    
    return docling_data, baml_data

# Load initial data
docling_data, baml_data = load_current_data()

In [ ]:
# Create an interactive button to load and display the selected example
load_button = widgets.Button(
    description='Load & Display Selected Example',
    button_style='success',
    layout=widgets.Layout(width='300px')
)

output_area = widgets.Output()

def on_load_click(b):
    """Handle load button click."""
    with output_area:
        clear_output(wait=True)
        
        if not current_paths:
            print("❌ No example selected")
            return
        
        print(f"🔄 Loading example: {example_dropdown.value}")
        
        # Load data
        global docling_data, baml_data
        docling_data, baml_data = load_current_data()
        
        # Display PDF
        print("\n📄 Displaying PDF page:")
        display_pdf_page(current_paths['pdf'])

load_button.on_click(on_load_click)

display(load_button)
display(output_area)

In [ ]:
def display_json_comparison(docling_data: Union[Dict, List], baml_data: Union[Dict, List]):
    """Display both JSON files side by side with full content."""
    
    # Handle empty data
    if not docling_data:
        docling_str = "No data available"
    else:
        docling_str = json.dumps(docling_data, indent=2, default=str)
    
    if not baml_data:
        baml_str = "No data available"
    else:
        baml_str = json.dumps(baml_data, indent=2, default=str)
    
    html = f"""
    <div style="display: flex; gap: 20px;">
        <div style="flex: 1; background-color: #f0f8ff; padding: 15px; border-radius: 5px;">
            <h3>🔍 Docling Output (Structural)</h3>
            <pre style="overflow-x: auto; max-height: 600px; font-size: 11px; white-space: pre-wrap;">{docling_str}</pre>
        </div>
        <div style="flex: 1; background-color: #f0fff0; padding: 15px; border-radius: 5px;">
            <h3>🤖 BAML Output (Semantic)</h3>
            <pre style="overflow-x: auto; max-height: 600px; font-size: 11px; white-space: pre-wrap;">{baml_str}</pre>
        </div>
    </div>
    """
    display(HTML(html))

# Create button to show comparison
compare_button = widgets.Button(
    description='Show JSON Comparison',
    button_style='info',
    layout=widgets.Layout(width='300px')
)

comparison_output = widgets.Output()

def on_compare_click(b):
    """Handle compare button click."""
    with comparison_output:
        clear_output(wait=True)
        
        if not current_paths:
            print("❌ No example selected. Please select an example first.")
            return
        
        print(f"📊 Comparing results for: {example_dropdown.value}")
        
        # Reload data to ensure we have the latest
        current_docling, current_baml = load_current_data()
        display_json_comparison(current_docling, current_baml)

compare_button.on_click(on_compare_click)

display(compare_button)
display(comparison_output)